# Análise dos indicadores (camada marts)

Lê as seis tabelas da camada marts, produzidas pelo `dbt build`, para a análise dos indicadores de desempenho e de absenteísmo crônico. As tabelas são lidas diretamente do BigQuery (`list_rows`, sem consulta SQL), sem gerar cópias em arquivo.

**Para rodar:**

1. Rode o `dbt build` (ver README), que cria as tabelas no dataset `dbt_dev_marts`.
2. Autentique: `gcloud auth application-default login`.
3. No ambiente do projeto: `pip install pandas pyarrow db-dtypes google-cloud-bigquery-storage ipykernel`.
4. Ajuste `PROJETO` na célula de variáveis e execute todas as células.

In [1]:
# Importações
import pandas as pd
from google.cloud import bigquery

In [2]:
# Variáveis Globais
PROJETO = "desafio-cientista-dados-peqc"  # ajuste para o seu projeto
DATASET = "dbt_dev_marts"

# Tamanho mínimo para comparar escolas e bairros: abaixo disso, um único caso
# muda a taxa ou a média em vários pontos (mesmos cortes do relatório).
MINIMO_ALUNOS = 30            # escolas, no absenteísmo
MINIMO_NOTAS = 30             # escolas e bairros, no desempenho, por disciplina
MINIMO_VINCULOS_BAIRRO = 300  # bairros, no absenteísmo

In [3]:
# Instância do cliente BigQuery
cliente = bigquery.Client(project=PROJETO)

In [4]:
# Funções auxiliares
def carregar(tabela):
    """Lê a tabela inteira do BigQuery (leitura direta, sem consulta SQL)."""
    return cliente.list_rows(f"{PROJETO}.{DATASET}.{tabela}").to_dataframe()

In [5]:
# Carga das tabelas da camada marts
desempenho_escola = carregar("mart_educacao__desempenho_escola")
desempenho_bairro = carregar("mart_educacao__desempenho_bairro")
desempenho_faixa_etaria = carregar("mart_educacao__desempenho_faixa_etaria")

absenteismo_escola = carregar("mart_educacao__absenteismo_escola")
absenteismo_bairro = carregar("mart_educacao__absenteismo_bairro")
absenteismo_disciplina = carregar("mart_educacao__absenteismo_disciplina")

In [6]:
# Conferência da carga: linhas e colunas de cada tabela
pd.DataFrame(
    {
        "qtd_linhas": [len(df) for df in (desempenho_escola, desempenho_bairro, desempenho_faixa_etaria,
                                          absenteismo_escola, absenteismo_bairro, absenteismo_disciplina)],
        "qtd_colunas": [df.shape[1] for df in (desempenho_escola, desempenho_bairro, desempenho_faixa_etaria,
                                               absenteismo_escola, absenteismo_bairro, absenteismo_disciplina)],
    },
    index=["desempenho_escola", "desempenho_bairro", "desempenho_faixa_etaria",
           "absenteismo_escola", "absenteismo_bairro", "absenteismo_disciplina"],
)

,qtd_linhas,qtd_colunas
desempenho_escola,1425,8
desempenho_bairro,402,8
desempenho_faixa_etaria,9,8
absenteismo_escola,521,6
absenteismo_bairro,135,6
absenteismo_disciplina,4,5


## Principais achados

Cada seção abaixo reproduz um achado da seção **Principais achados** do [README](../README.md), a partir dos marts carregados acima. Os dados foram aleatorizados na anonimização, então os números ilustram o método e não descrevem a rede real.

### 1. Absenteísmo por escola: o problema está concentrado em poucas escolas

A taxa da rede soma numeradores e denominadores de todas as escolas (não é a média das taxas). A distribuição e a concentração usam só as escolas com `MINIMO_ALUNOS` ou mais, porque em escolas pequenas um único caso muda a taxa em vários pontos.

In [7]:
taxa_rede = (
    absenteismo_escola["qtd_alunos_absenteismo_cronico"].sum()
    / absenteismo_escola["qtd_alunos"].sum()
)
escolas_comparaveis = absenteismo_escola[absenteismo_escola["qtd_alunos"] >= MINIMO_ALUNOS]
quantis = escolas_comparaveis["taxa_absenteismo_cronico"].quantile([0.25, 0.5, 0.75, 0.9])

pd.Series({
    "vinculos_aluno_escola": absenteismo_escola["qtd_alunos"].sum(),
    "vinculos_abaixo_de_75": absenteismo_escola["qtd_alunos_absenteismo_cronico"].sum(),
    "taxa_rede_pct": round(100 * taxa_rede, 2),
    "escolas_comparaveis": len(escolas_comparaveis),
    "p25_pct": round(100 * quantis[0.25], 1),
    "mediana_pct": round(100 * quantis[0.5], 1),
    "p75_pct": round(100 * quantis[0.75], 1),
    "p90_pct": round(100 * quantis[0.9], 1),
    "maximo_pct": round(100 * escolas_comparaveis["taxa_absenteismo_cronico"].max(), 1),
})

vinculos_aluno_escola    99032.00
vinculos_abaixo_de_75     3177.00
taxa_rede_pct                3.21
escolas_comparaveis        475.00
p25_pct                      0.00
mediana_pct                  1.70
p75_pct                      4.70
p90_pct                      7.60
maximo_pct                  22.80
dtype: float64

**Concentração:** ordenando as escolas pelo número de casos, quantas reúnem metade deles e que parcela dos alunos elas atendem.

In [8]:
ordenadas = escolas_comparaveis.sort_values("qtd_alunos_absenteismo_cronico", ascending=False)
casos_acumulados = (
    ordenadas["qtd_alunos_absenteismo_cronico"].cumsum()
    / ordenadas["qtd_alunos_absenteismo_cronico"].sum()
)
qtd_escolas_metade = int((casos_acumulados < 0.5).sum()) + 1

pd.Series({
    "escolas_com_metade_dos_casos": qtd_escolas_metade,
    "pct_dos_alunos_nessas_escolas": round(
        100 * ordenadas["qtd_alunos"].iloc[:qtd_escolas_metade].sum() / ordenadas["qtd_alunos"].sum(), 1
    ),
    "escolas_sem_nenhum_caso": int((absenteismo_escola["qtd_alunos_absenteismo_cronico"] == 0).sum()),
    "escolas_total": len(absenteismo_escola),
})

escolas_com_metade_dos_casos      58.0
pct_dos_alunos_nessas_escolas     20.3
escolas_sem_nenhum_caso          156.0
escolas_total                    521.0
dtype: float64

### 2. Absenteísmo por bairro: a variação entre bairros é menor que entre escolas

Compara a faixa entre os percentis 10 e 90 dos bairros (com `MINIMO_VINCULOS_BAIRRO` ou mais) com a das escolas comparáveis. Se a faixa dos bairros for mais estreita, a agregação dilui as diferenças e a escola é a unidade mais útil para intervir.

In [9]:
bairros_comparaveis = absenteismo_bairro[
    absenteismo_bairro["qtd_vinculos_aluno_escola"] >= MINIMO_VINCULOS_BAIRRO
]

display(pd.DataFrame({
    "bairros": 100 * bairros_comparaveis["taxa_absenteismo_cronico"].quantile([0.1, 0.9]),
    "escolas": 100 * escolas_comparaveis["taxa_absenteismo_cronico"].quantile([0.1, 0.9]),
}).round(1).rename(index={0.1: "p10_pct", 0.9: "p90_pct"}))

pd.Series({
    "bairros_total": len(absenteismo_bairro),
    "bairros_comparaveis": len(bairros_comparaveis),
    "bairros_com_uma_escola": int((absenteismo_bairro["qtd_escolas"] == 1).sum()),
})

,bairros,escolas
p10_pct,1.1,0.0
p90_pct,5.9,7.6


bairros_total             135
bairros_comparaveis        81
bairros_com_uma_escola     43
dtype: int64

### 3. Absenteísmo por disciplina: Português e Matemática bem acima de Ciências e Inglês

A coluna `razao_sobre_ciencias` mostra quantas vezes a taxa de cada disciplina é a de Ciências. A taxa global vem do item 1: as taxas por disciplina ficam acima dela porque a global soma faltas e dias de todas as disciplinas.

In [10]:
por_disciplina = absenteismo_disciplina.set_index("disciplina")[
    ["qtd_vinculos_aluno_escola", "qtd_vinculos_absenteismo_cronico", "taxa_absenteismo_cronico"]
].sort_values("taxa_absenteismo_cronico", ascending=False)

por_disciplina["taxa_pct"] = (100 * por_disciplina["taxa_absenteismo_cronico"]).round(1)
por_disciplina["razao_sobre_ciencias"] = (
    por_disciplina["taxa_absenteismo_cronico"] / por_disciplina.loc["ciencias", "taxa_absenteismo_cronico"]
).round(1)
por_disciplina["acima_da_taxa_global"] = por_disciplina["taxa_absenteismo_cronico"] > taxa_rede
por_disciplina

,qtd_vinculos_aluno_escola,qtd_vinculos_absenteismo_cronico,taxa_absenteismo_cronico,taxa_pct,razao_sobre_ciencias,acima_da_taxa_global
disciplina,,,,,,
portugues,99020,9950,0.100485,10.0,2.9,True
matematica,99017,9688,0.097842,9.8,2.8,True
ciencias,99029,3401,0.034343,3.4,1.0,True
ingles,99029,680,0.006867,0.7,0.2,False


### 4. Desempenho por escola: Matemática tem a menor média e a associação com o absenteísmo é fraca

As médias da rede vêm do mart por faixa etária, que tem todas as notas (com e sem escola conhecida), ponderadas pelo número de notas. A dispersão entre escolas usa só as escolas com `MINIMO_NOTAS` ou mais na disciplina.

In [11]:
rede = desempenho_faixa_etaria.groupby("disciplina")[["soma_notas", "qtd_notas"]].sum()
rede["media_rede"] = (rede["soma_notas"] / rede["qtd_notas"]).round(2)

escolas_com_notas = desempenho_escola[desempenho_escola["qtd_notas"] >= MINIMO_NOTAS]
dispersao = escolas_com_notas.groupby("disciplina")["media_nota"].quantile([0.1, 0.5, 0.9]).unstack()
dispersao.columns = ["p10_escolas", "mediana_escolas", "p90_escolas"]

resumo_desempenho = rede[["qtd_notas", "media_rede"]].join(dispersao.round(2))
resumo_desempenho["escolas_comparaveis"] = escolas_com_notas.groupby("disciplina").size()
resumo_desempenho["distancia_p90_p10"] = (
    resumo_desempenho["p90_escolas"] - resumo_desempenho["p10_escolas"]
).round(2)
resumo_desempenho.sort_values("media_rede")

,qtd_notas,media_rede,p10_escolas,mediana_escolas,p90_escolas,escolas_comparaveis,distancia_p90_p10
disciplina,,,,,,,
matematica,185883,5.82,4.96,5.93,6.71,364,1.75
ciencias,185420,6.23,5.35,6.29,7.15,364,1.80
portugues,186736,6.44,5.65,6.57,7.33,364,1.68


**Matemática é a menor em todos os recortes?** Para cada faixa etária e para cada estatística entre escolas, a disciplina com o menor valor.

In [12]:
menor_por_faixa = (
    desempenho_faixa_etaria.loc[
        desempenho_faixa_etaria.groupby("faixa_etaria")["media_nota"].idxmin(),
        ["ordem_faixa_etaria", "faixa_etaria", "disciplina"],
    ]
    .sort_values("ordem_faixa_etaria")
    .set_index("faixa_etaria")["disciplina"]
)
menor_entre_escolas = resumo_desempenho[["media_rede", "p10_escolas", "mediana_escolas", "p90_escolas"]].idxmin()

pd.concat([menor_por_faixa, menor_entre_escolas]).rename("disciplina_com_menor_valor").to_frame()

,disciplina_com_menor_valor
11-14,matematica
15-17,matematica
18+,matematica
media_rede,matematica
p10_escolas,matematica
mediana_escolas,matematica
p90_escolas,matematica


**Absenteísmo × desempenho:** as duas medidas vêm de fontes diferentes e só se encontram na escola. A média das notas da escola soma as três disciplinas (ponderada pelas notas). Entram as escolas com `MINIMO_ALUNOS` ou mais na frequência e `MINIMO_NOTAS` ou mais na avaliação. A correlação de Spearman (por postos) confere se o resultado depende de valores extremos.

In [13]:
notas_por_escola = desempenho_escola.groupby("id_escola")[["soma_notas", "qtd_notas"]].sum()
notas_por_escola["media_notas"] = notas_por_escola["soma_notas"] / notas_por_escola["qtd_notas"]

cruzamento = absenteismo_escola.set_index("id_escola").join(notas_por_escola, how="inner")
cruzamento = cruzamento[
    (cruzamento["qtd_alunos"] >= MINIMO_ALUNOS) & (cruzamento["qtd_notas"] >= MINIMO_NOTAS)
]

pd.Series({
    "escolas": len(cruzamento),
    "correlacao_pearson": round(cruzamento["taxa_absenteismo_cronico"].corr(cruzamento["media_notas"]), 2),
    # Spearman = Pearson sobre os postos (dispensa o scipy)
    "correlacao_spearman": round(
        cruzamento["taxa_absenteismo_cronico"].rank().corr(cruzamento["media_notas"].rank()), 2
    ),
})

escolas                369.00
correlacao_pearson      -0.26
correlacao_spearman     -0.23
dtype: float64

### 5. Desempenho por bairro: a agregação dilui as diferenças entre escolas

Mesma comparação do item 2, agora para as médias: a faixa entre os percentis 10 e 90 dos bairros com `MINIMO_NOTAS` ou mais na disciplina, ao lado da faixa das escolas.

In [14]:
bairros_com_notas = desempenho_bairro[desempenho_bairro["qtd_notas"] >= MINIMO_NOTAS]
faixa_bairros = bairros_com_notas.groupby("disciplina")["media_nota"].quantile([0.1, 0.9]).unstack()
faixa_bairros.columns = ["p10_bairros", "p90_bairros"]

comparacao = faixa_bairros.join(resumo_desempenho[["p10_escolas", "p90_escolas"]]).round(2)
comparacao["distancia_bairros"] = (comparacao["p90_bairros"] - comparacao["p10_bairros"]).round(2)
comparacao["distancia_escolas"] = (comparacao["p90_escolas"] - comparacao["p10_escolas"]).round(2)
comparacao["bairros_comparaveis"] = bairros_com_notas.groupby("disciplina").size()
comparacao

,p10_bairros,p90_bairros,p10_escolas,p90_escolas,distancia_bairros,distancia_escolas,bairros_comparaveis
disciplina,,,,,,,
ciencias,5.61,6.86,5.35,7.15,1.25,1.80,127
matematica,5.20,6.55,4.96,6.71,1.35,1.75,127
portugues,5.79,7.04,5.65,7.33,1.25,1.68,127


### 6. Desempenho por faixa etária: 15 a 17 anos com médias mais altas e menor dispersão

Médias e desvios padrão por faixa e disciplina. As faixas que não aparecem na tabela não têm nenhuma nota na `avaliacao`. Lembrete do perfilamento: a faixa etária da tabela `aluno` tem distribuição implausível para a rede municipal, então as diferenças entre faixas pedem cautela.

In [15]:
por_faixa = desempenho_faixa_etaria.sort_values(["ordem_faixa_etaria", "disciplina"])

display(por_faixa.pivot(index="faixa_etaria", columns="disciplina", values="media_nota")
        .loc[por_faixa["faixa_etaria"].unique()].round(2))
display(por_faixa.pivot(index="faixa_etaria", columns="disciplina", values="desvio_padrao_nota")
        .loc[por_faixa["faixa_etaria"].unique()].round(2))

pd.Series({
    "faixas_com_notas": ", ".join(por_faixa["faixa_etaria"].unique()),
    "faixa_com_maior_media_em_todas_as_disciplinas": ", ".join(
        por_faixa.loc[por_faixa.groupby("disciplina")["media_nota"].idxmax(), "faixa_etaria"].unique()
    ),
    "faixa_com_menor_desvio_em_todas_as_disciplinas": ", ".join(
        por_faixa.loc[por_faixa.groupby("disciplina")["desvio_padrao_nota"].idxmin(), "faixa_etaria"].unique()
    ),
})

disciplina,ciencias,matematica,portugues
faixa_etaria,,,
11-14,5.58,5.27,5.68
15-17,6.30,5.87,6.52
18+,5.45,5.27,5.39


disciplina,ciencias,matematica,portugues
faixa_etaria,,,
11-14,2.76,2.58,2.73
15-17,2.13,1.94,1.91
18+,2.87,2.68,2.74


faixas_com_notas                                  11-14, 15-17, 18+
faixa_com_maior_media_em_todas_as_disciplinas                 15-17
faixa_com_menor_desvio_em_todas_as_disciplinas                15-17
dtype: str